# Single-seed preliminary comparison

This is a single-seed preliminary scope: it runs raw-direct first and then candidate CNN, for 3 hours each. Plan for roughly 6 hours plus setup/eval. It is not statistically conclusive and must not be interpreted as a general CNN claim.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)


In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError("This overnight comparison requires a CUDA-enabled Colab runtime.")
subprocess.run(["nvidia-smi"], check=True)
ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
print(f"Torch: {torch.__version__} ({torch.__file__})")
print(f"CUDA available: {torch.cuda.is_available()}, CUDA: {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"RAM: {ram_bytes / 2**30:.2f} GiB")


In [ ]:
from pathlib import Path

repo_url = "https://github.com/LMS4681/CNN-RL-Raw-Comparison.git"
tag = "deadline-preview-20260721-r2"
target = Path("/content/CNN-RL-Raw-Comparison")
if target.exists():
    if not target.is_dir() or target.resolve() != target or target.parent != Path("/content"):
        raise RuntimeError(f"Refusing to reuse anything except the exact checkout path: {target}")
    remote = subprocess.run(["git", "-C", str(target), "remote", "get-url", "origin"], text=True, capture_output=True, check=True).stdout.strip()
    if remote != repo_url:
        raise RuntimeError(f"Refusing unexpected checkout origin: {remote!r}")
    if subprocess.run(["git", "-C", str(target), "status", "--porcelain"], text=True, capture_output=True, check=True).stdout.strip():
        raise RuntimeError("Existing comparison checkout is dirty; do not overwrite it.")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin", "tag", tag], check=True)
    subprocess.run(["git", "-C", str(target), "checkout", "--detach", tag], check=True)
else:
    subprocess.run(["git", "clone", "--branch", tag, "--depth", "1", repo_url, str(target)], check=True)
head = subprocess.run(["git", "-C", str(target), "rev-parse", "HEAD"], text=True, capture_output=True, check=True).stdout.strip()
resolved_tag = subprocess.run(["git", "-C", str(target), "describe", "--exact-match", "--tags", "HEAD"], text=True, capture_output=True, check=True).stdout.strip()
if resolved_tag != tag:
    raise RuntimeError(f"Checkout is not the required immutable tag {tag}: {resolved_tag!r}")
if subprocess.run(["git", "-C", str(target), "status", "--porcelain"], text=True, capture_output=True, check=True).stdout.strip():
    raise RuntimeError("Comparison checkout is dirty after tag checkout.")
Path("/content/CNN-RL-Raw-Comparison.overnight-v1.head").write_text(head + "\n", encoding="utf-8")
print(f"overnight-v1 HEAD: {head}")


In [ ]:
import json
import sys

snapshot_script = r'''
import importlib.metadata
import json
import pathlib
import torch

distributions = {}
for distribution in importlib.metadata.distributions():
    name = (distribution.metadata.get("Name") or "").lower()
    if name == "torch" or name == "triton" or name.startswith("nvidia-"):
        distributions[name] = {
            "name": name,
            "version": distribution.version,
            "location": str(pathlib.Path(distribution.locate_file("")).resolve()),
        }
print(json.dumps({
    "torch": {
        "version": torch.__version__,
        "path": str(pathlib.Path(torch.__file__).resolve()),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
    },
    "distributions": distributions,
}, sort_keys=True))
'''

def child_torch_snapshot():
    result = subprocess.run(
        [sys.executable, "-c", snapshot_script],
        text=True,
        capture_output=True,
        check=True,
    )
    return json.loads(result.stdout)

before_snapshot = child_torch_snapshot()
before_torch = before_snapshot["torch"]
before_distributions = before_snapshot["distributions"]
lock_path = target / "AllocRL" / "requirements-comparison.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--require-hashes", "-r", str(lock_path)], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
after_snapshot = child_torch_snapshot()
after_torch = after_snapshot["torch"]
after_distributions = after_snapshot["distributions"]
assert after_torch == before_torch, f"Hashed dependency install changed Colab Torch/CUDA: {before_torch} -> {after_torch}"
assert after_distributions == before_distributions, f"Hashed dependency install changed Torch/CUDA distributions: {before_distributions} -> {after_distributions}"


In [ ]:
%cd /content/CNN-RL-Raw-Comparison/AllocRL

import hashlib
import json
import re

expected = {
    "baseline_commit": "cd4e14fc1725a4ff159e59d6874d3602f3b65a06",
    "fixed_scenarios_sha256": "6125f53939a1b8eef8662b2628c0da2f1d0f26b5b541a99252858326b38cd814",
    "split_manifest_sha256": "d3df1d0076248b4bcbddb4c910a3cb81481da65c7415c6b3cacf9e055cc3f9df",
}
config = json.loads(Path("configs/overnight_seed0.json").read_text(encoding="utf-8"))
for key, value in expected.items():
    if config.get(key) != value:
        raise RuntimeError(f"Immutable config mismatch for {key}: {config.get(key)!r}")
lock_sha256 = hashlib.sha256(Path("requirements-comparison.txt").read_bytes()).hexdigest()
provenance = Path("../UPSTREAM_BASELINE.md").read_text(encoding="utf-8")
match = re.search(r"Dependency lock SHA256: `([0-9a-f]{64})`", provenance)
if match is None or match.group(1) != lock_sha256:
    raise RuntimeError("requirements-comparison.txt SHA-256 does not match UPSTREAM_BASELINE.md")
print(f"Verified comparison lock SHA-256: {lock_sha256}")


In [ ]:
output_root = "/content/drive/MyDrive/CNN-RL-comparison/overnight-20260721"
rerun_command = "python -m comparison.experiment_runner --config ./configs/overnight_seed0.json --output-root /content/drive/MyDrive/CNN-RL-comparison/overnight-20260721 --take-over-stale-lease"
print(rerun_command)
runner_error = None
try:
    subprocess.run([sys.executable, "-m", "comparison.experiment_runner", "--config", "./configs/overnight_seed0.json", "--output-root", output_root, "--take-over-stale-lease"], check=True)
except subprocess.CalledProcessError as error:
    runner_error = error


In [ ]:
from IPython.display import Markdown, display

artifact_root = Path("/content/drive/MyDrive/CNN-RL-comparison/overnight-20260721")
complete = artifact_root / "COMPLETE.json"
korean_report = artifact_root / "comparison" / "preliminary_comparison_ko.md"
partial = artifact_root / "comparison" / "PARTIAL_REPORT.md"
if complete.is_file() and korean_report.is_file():
    display(Markdown(f"## COMPLETE\n\nMarker: `{complete}`"))
    display(Markdown(korean_report.read_text(encoding="utf-8")))
else:
    display(Markdown("## PARTIAL"))
    if partial.is_file():
        display(Markdown(partial.read_text(encoding="utf-8")))
    display(Markdown("Rerun all (or resume manually) with:\n\n```bash\npython -m comparison.experiment_runner --config ./configs/overnight_seed0.json --output-root /content/drive/MyDrive/CNN-RL-comparison/overnight-20260721 --take-over-stale-lease\n```"))
if runner_error is not None:
    raise RuntimeError("The experiment runner failed; the COMPLETE/PARTIAL status above is the on-Drive state.") from runner_error
